![JupyterLite HIV Drug Resistance Maps: an original conceptual globe, map, and molecular illustration.](assets/hivdr-splash.png)

**[Open this notebook in JupyterLite Lab](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/lab/index.html?path=09_study_queries_and_provenance.ipynb)** · **[Read the Jupyter Book](https://jltobias.github.io/JupyterLite-HIVDB-Drug-Resistance-Maps/book/)** · [Map and globe resources](mapping_resources.md)

Original conceptual artwork; its markers and bars are not observations.

# Study queries and provenance

Project: JupyterLite HIV Drug Resistance Maps. Created/updated: 2026-11-16.
Author: [Repository contributors; generated with Codex]. Email/organization: [Not supplied].
Input: factual HIVDB metadata snapshot in `data/studies/`, plus the source-checked annotation ledger.
Output: computed pandas tables and Plotly figures; optional exports under `outputs/`.
Runtime: Python 3.11+, pandas, Plotly 6.2.0, IPython; native retrieval dependencies are listed in `study_query.py`.
Change log: initial reproducible study-inventory notebook. No simulated study data.

**Unit:** a distinct HIVDB reference/publication ID. A publication can describe multiple study components; multiple publications can describe one cohort. This is a descriptive inventory, not a population sample, national resistance estimate, or deduplicated systematic review.

Follow the chain **public calls → factual metadata → reference-country keys → domain matrix**.

The snapshot combines the public reference index, PR/RT/IN GenoRx downloads, the phenotype reference table, and two explicit live HIV1 group M queries: prior DTG exposure (IN) and prior LEN exposure (CA). This is broad coverage of selected HIVDB collections, **not an export of every database table**. The older bulk downloads and current API have different update schedules. Sierra is not used for this inventory.

Original articles, abstracts, sequences, patient identifiers, website images and Stanford logos are not bundled. `annotations.csv` contains our factual classifications with links; `signals.csv` contains unverified title cues with explicit matching rules.

In [ ]:
# Read immutable factual inputs and check their recorded hashes before analysis.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Code, HTML, FileLink
from study_matrix import (load_snapshot, country_membership, build_domains,
                          country_matrix, coverage_table, country_summary,
                          evidence_ledger, DOMAINS, UNKNOWN)
catalogue, evidence, signals, annotations, manifest = load_snapshot()
px.defaults.template = "plotly_white"
print("Snapshot built:", manifest["built_utc"])
print("Catalogue publication records:", len(catalogue))

In [ ]:
# The request manifest records exact filters, pagination totals and API/database versions.
query_audit = pd.DataFrame([
    {"Drug": q["drug"], "Gene": q["gene"], "References": q["totals"]["reference_count"],
     "Selected isolates": q["totals"]["isolates"], "Pages": q["pages"],
     "API version": q["api_version"], "Database version": q["database_version"]}
    for q in manifest["queries"]
])
display(query_audit)
display(pd.DataFrame(manifest["genorx"]).T.rename_axis("Bulk download"))

## Inspect the actual query code

These functions below come from the delivered, runnable [study_query.py](study_query.py). The POST endpoint performs a read-only search. Every page is retrieved; totals and database version must remain consistent. One source-form request per reference retrieves sampling geography for **all selected isolate IDs**, avoiding a one-isolate-per-study shortcut. Repeated display labels and clone rows are handled before a checked one-to-one merge.

The `.py` file also contains the reference HTML parser, bulk TSV calls, exact country mapping, timestamp/hash receipts, and explicit failure checks. It is the single implementation called by `scripts/fetch_study_metadata.py`.

In [ ]:
# Show the real source without importing native-only network/parser dependencies.
import ast
query_source = Path("study_query.py").read_text(encoding="utf-8")
tree = ast.parse(query_source)
for node in tree.body:
    if isinstance(node, ast.FunctionDef) and node.name in {"query_isolates", "sampling_metadata"}:
        display(Code(ast.get_source_segment(query_source, node), language="python"))

## Reproduce or refresh

The packaged snapshot makes **Run All** independent of Stanford's availability and browser CORS restrictions. It is not a silent fallback: the timestamp above identifies exactly what was loaded. To refresh locally, clone this repository, install `requirements.txt`, and run:

```bash
python scripts/fetch_study_metadata.py --refresh
```

Omit `--refresh` to replay the source cache and complete any interrupted retrieval. The raw cache remains under ignored `_build/study-source-cache/`; only factual metadata and audit receipts are published. Native code fails explicitly on changed schemas, missing pages, ambiguous joins or a changing database. The optional cell below runs the same native pipeline; leave it off in JupyterLite.

In [ ]:
# Optional network refresh: explicit opt-in, never a hidden call during Book/Lab execution.
REFRESH_FROM_STANFORD = False
if REFRESH_FROM_STANFORD:
    import sys, subprocess
    if sys.platform in {"emscripten", "wasi"}:
        raise RuntimeError("Refresh in native Python using the repository CLI; JupyterLite uses the shipped snapshot.")
    subprocess.run([sys.executable, "study_query.py", "--refresh"], check=True)
else:
    print("Using the dated, hash-checked snapshot. No Stanford calls were made by this notebook.")

## Audit the geography and collection years

Geography is the recorded sampling country/area, not author affiliation or every trial site. Ambiguous labels remain unmapped. `< 2003` means an upper bound and must not become a point at 2003. Trial enrollment periods, publication years and exact specimen years remain different fields.

In [ ]:
# Preserve each unresolved source label for review instead of guessing a country.
unmapped = (evidence.loc[evidence.iso3.eq("")].groupby("country_raw")
            .agg(Publications=("ref_id", "nunique"), Metadata_rows=("ref_id", "size")))
display(unmapped)
year_audit = evidence.groupby("collection_year_status").ref_id.nunique().rename("Publications").reset_index()
display(year_audit)
# A publication can contain both exact and bounded years: these rows overlap.
fig = px.bar(year_audit, x="collection_year_status", y="Publications",
             title="Sampling-year information (publication categories overlap)",
             labels={"collection_year_status":"Source year format"}, color_discrete_sequence=["#087e8b"])
fig.show()

In [ ]:
# A disjoint flow of catalogue records; each RefID follows exactly one branch.
mapped_ids = set(evidence.loc[evidence.iso3.ne(""), "ref_id"])
any_geo_ids = set(evidence.ref_id)
counts = [len(mapped_ids), len(any_geo_ids - mapped_ids), len(catalogue) - len(any_geo_ids)]
assert sum(counts) == len(catalogue)
fig = go.Figure(go.Sankey(node=dict(label=["Catalogue publications", "At least one mapped country",
    "Evidence, but geography unmapped", "No selected-source sampling metadata"],
    color=["#183153", "#087e8b", "#e8a443", "#bcc6d2"]),
    link=dict(source=[0,0,0], target=[1,2,3], value=counts)))
fig.update_layout(title="What can be placed on the country matrix?", height=400)
fig.show()

## Source links

[HIVDB reference index](https://hivdb.stanford.edu/cgi-bin/Reference.cgi), [GenoRx downloads and field definitions](https://hivdb.stanford.edu/pages/geno-rx-datasets.html), [phenotype datasets](https://hivdb.stanford.edu/pages/genopheno.dataset.html), [integrase query](https://hivdb.stanford.edu/HIVDB3/inhibitors-mutations/IN/), [capsid query](https://hivdb.stanford.edu/HIVDB3/inhibitors-mutations/CA/), [database citation](https://hivdb.stanford.edu/pages/citation.html), [terms and FAQs](https://hivdb.stanford.edu/pages/FAQ/FAQ.html).

Next: [build and inspect the country matrix](10_country_study_matrix.ipynb).